# Ex-Ante Embedding Recovery of Rate Sensitivity

This notebook tests whether holdings embeddings formed before a quarter recover firms' historical rate sensitivities and predict their reactions to FOMC announcements during that quarter. It distinguishes economic timing from strict public-data timing and never uses post-announcement holdings for prediction.

## 1. Setup and frozen timing rules

Economic timing uses q−1 quarter-end holdings: these positions existed before events in q. Public-availability timing uses q−2 holdings, conservatively allowing for the reporting delay. A later notebook will use q+1 only to study post-shock rebalancing.

In [ ]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
from IPython.display import display

def locate_root():
    start=Path.cwd().resolve()
    for candidate in [start,*start.parents]:
        if (candidate/'economic_shock_sensitivity').exists() and (candidate/'full_replication_80q').exists(): return candidate
    raise FileNotFoundError('Could not locate Asset_Embeddings root.')

ROOT=locate_root(); MODULE=ROOT/'economic_shock_sensitivity'; RESULTS=MODULE/'results'/'ex_ante_embedding_models'
RESULTS.mkdir(parents=True,exist_ok=True); sys.path.insert(0,str(MODULE/'src'))
import embedding_rate_models as erm
import future_target_validation as ftv
measurement_gate=pd.read_parquet(MODULE/'results'/'future_target_validation'/'measurement_validity_gate.parquet')
display(measurement_gate); assert measurement_gate.passed.all()
targets=pd.read_parquet(MODULE/'data'/'processed'/'rolling_rate_targets'/'all_rolling_rate_targets.parquet')
targets=targets[(targets.specification.eq('stmt_ar00'))&targets.eligible].copy()
quarters=sorted(targets.anchor_quarter.unique())
print('Quarters:',len(quarters),quarters[0],quarters[-1])


## 2. Model inventory

RS-Binary and RS-L-Min are tested at four and ten dimensions. Characteristics provide a conventional benchmark, and combined specifications ask whether embeddings add information beyond them. Every model is evaluated under both timing conventions.

In [ ]:
base_specs=[
 {'base_name':'rs_binary_4','model':'rs_binary','dimension':4,'add_characteristics':False,'characteristics_only':False},
 {'base_name':'rs_binary_10','model':'rs_binary','dimension':10,'add_characteristics':False,'characteristics_only':False},
 {'base_name':'rs_l_min_4','model':'rs_l_min','dimension':4,'add_characteristics':False,'characteristics_only':False},
 {'base_name':'rs_l_min_10','model':'rs_l_min','dimension':10,'add_characteristics':False,'characteristics_only':False},
 {'base_name':'characteristics_6','model':'rs_binary','dimension':0,'add_characteristics':True,'characteristics_only':True},
 {'base_name':'rs_binary_10_plus_chars','model':'rs_binary','dimension':10,'add_characteristics':True,'characteristics_only':False},
 {'base_name':'rs_l_min_10_plus_chars','model':'rs_l_min','dimension':10,'add_characteristics':True,'characteristics_only':False}]
model_specs=[]
for lag,timing in [(1,'economic_q_minus_1'),(2,'public_q_minus_2')]:
    for spec in base_specs:
        item=dict(spec); item.update(embedding_lag=lag,timing=timing,name=f"{spec['base_name']}__{timing}"); model_specs.append(item)
display(pd.DataFrame(model_specs))


## 3. Load quarterly design matrices

The permanent firm split created before model fitting is retained in every quarter. Missing characteristics are imputed later using training-firm medians only.

In [ ]:
datasets={}; inventory=[]
for spec in model_specs:
    for quarter in quarters:
        frame,features,input_quarter=erm.load_quarter_frame(ROOT,targets,quarter,spec['model'],spec['dimension'],spec['embedding_lag'],spec['add_characteristics'],spec['characteristics_only'])
        datasets[(spec['name'],quarter)]=(frame,features)
        inventory.append({'model':spec['name'],'quarter':quarter,'input_quarter':input_quarter,'firms':len(frame),'features':len(features),'train_firms':frame.firm_split.eq('train').sum(),'validation_firms':frame.firm_split.eq('validation').sum(),'test_firms':frame.firm_split.eq('test').sum()})
inventory=pd.DataFrame(inventory); inventory.to_parquet(RESULTS/'model_input_inventory.parquet',index=False)
display(inventory.groupby('model',as_index=False).agg(quarters=('quarter','nunique'),minimum_firms=('firms','min'),median_firms=('firms','median'),features=('features','max')))


## 4. Select ridge penalties using development quarters only

For each model, ridge penalties are compared using validation firms in 2010–2016. Feature scaling, imputation, and target winsorization are fitted only on training firms within each quarter. No 2017–2024 outcome influences penalty selection.

In [ ]:
alpha_grid=np.array([0.01,0.1,1,10,100,1000,10000],float)
cv_rows=[]
development=[q for q in quarters if int(q[:4])<=2016]
for number,spec in enumerate(model_specs,1):
    for alpha in alpha_grid:
        summaries=[]
        for quarter in development:
            frame,features=datasets[(spec['name'],quarter)]
            _,summary=erm.fit_predict(frame,features,alpha,fit_splits=('train',),predict_split='validation')
            summary.update(quarter=quarter); summaries.append(summary)
        summary_frame=pd.DataFrame(summaries)
        cv_rows.append({'model':spec['name'],'alpha':alpha,'development_quarters':len(summaries),'pooled_validation_variance_r2':erm.pooled_variance_r2(summary_frame),'mean_quarter_validation_r2':summary_frame.variance_r2.mean(),'mean_validation_correlation':summary_frame.correlation.mean()})
    print(f'{number:02d}/{len(model_specs)} {spec["name"]}: CV complete')
cv=pd.DataFrame(cv_rows)
selected=(cv.sort_values(['model','pooled_validation_variance_r2'],ascending=[True,False]).groupby('model',as_index=False).first()[['model','alpha','pooled_validation_variance_r2']].rename(columns={'alpha':'selected_alpha','pooled_validation_variance_r2':'selection_r2'}))
cv.to_parquet(RESULTS/'development_alpha_curves.parquet',index=False); selected.to_parquet(RESULTS/'selected_penalties.parquet',index=False)
display(selected.sort_values('selection_r2',ascending=False))


## 5. Generate permanent-firm out-of-sample predictions

After selecting penalties, each quarterly model is refitted on train plus validation firms and predicts only permanent test firms. Test-firm targets never enter preprocessing, winsorization, or model fitting.

In [ ]:
prediction_frames=[]; fit_rows=[]
alpha_lookup=selected.set_index('model').selected_alpha.to_dict()
for number,spec in enumerate(model_specs,1):
    alpha=alpha_lookup[spec['name']]
    for quarter in quarters:
        frame,features=datasets[(spec['name'],quarter)]
        prediction,summary=erm.fit_predict(frame,features,alpha,fit_splits=('train','validation'),predict_split='test')
        prediction['model']=spec['name']; prediction['timing']=spec['timing']; prediction_frames.append(prediction)
        summary.update(model=spec['name'],timing=spec['timing'],quarter=quarter,alpha=alpha); fit_rows.append(summary)
    print(f'{number:02d}/{len(model_specs)} {spec["name"]}: test predictions complete')
predictions=pd.concat(prediction_frames,ignore_index=True); fits=pd.DataFrame(fit_rows)
predictions.to_parquet(RESULTS/'permanent_test_firm_predictions.parquet',index=False); fits.to_parquet(RESULTS/'quarterly_test_metrics.parquet',index=False)
display(predictions.head())


## 6. Cross-sectional target-recovery results

Variance R² uses the corresponding train-plus-validation mean as the benchmark. Metrics are pooled by squared-error sums rather than giving tiny quarterly samples disproportionate weight.

In [ ]:
fits['time_sample']=np.select([fits.quarter.str[:4].astype(int).le(2016),fits.quarter.str[:4].astype(int).le(2020)],['development','validation'],default='final_test')
summary_rows=[]
for (model,sample),group in fits.groupby(['model','time_sample']):
    summary_rows.append({'model':model,'time_sample':sample,'quarters':group.quarter.nunique(),'test_firm_observations':group.prediction_firms.sum(),'pooled_variance_r2':erm.pooled_variance_r2(group),'mean_quarter_variance_r2':group.variance_r2.mean(),'median_quarter_variance_r2':group.variance_r2.median(),'mean_prediction_correlation':group.correlation.mean()})
model_summary=pd.DataFrame(summary_rows); model_summary.to_parquet(RESULTS/'target_recovery_summary.parquet',index=False)
display(model_summary.sort_values(['time_sample','pooled_variance_r2'],ascending=[True,False]))


## 7. Do embedding-predicted betas forecast actual future event reactions?

Test-firm beta predictions are joined to FOMC events in the same target quarter. The prediction is multiplied by the realized statement surprise, and the same event-fixed-effect interaction test used for the historical target is repeated.

In [ ]:
event_returns=pd.read_parquet(MODULE/'data'/'processed'/'fomc_event_returns'/'firm_fomc_event_returns_2005_2024.parquet')
event_returns['event_date']=pd.to_datetime(event_returns.event_date); event_returns=event_returns[event_returns.event_date.ge('2010-01-01')].copy(); event_returns['anchor_quarter']=event_returns.event_date.dt.to_period('Q').astype(str).str.lower()
future_rows=[]
for model,model_predictions in predictions.groupby('model'):
    panel=event_returns.merge(model_predictions[['anchor_quarter','permco','prediction']],on=['anchor_quarter','permco'],how='inner',validate='many_to_one')
    panel['predicted_reaction']=panel.prediction*panel.shock_stmt
    panel['time_sample']=np.select([panel.event_date.dt.year.le(2016),panel.event_date.dt.year.le(2020)],['development','validation'],default='final_test')
    for sample in ['all','development','validation','final_test']:
        subset=panel if sample=='all' else panel[panel.time_sample.eq(sample)]
        result=ftv.interaction_regression(subset,'abnormal_return_00','predicted_reaction'); result.update(model=model,time_sample=sample); future_rows.append(result)
future_results=pd.DataFrame(future_rows); future_results.to_parquet(RESULTS/'embedding_predicted_beta_future_event_tests.parquet',index=False)
display(future_results.sort_values(['time_sample','interaction_slope'],ascending=[True,False]))


## 8. Predeclared decision table

The primary representation is ten-dimensional RS-Binary with q−1 economic timing. We require positive final-test target R², improvement over characteristics in the final test, and a positive future-event interaction. The q−2 specification determines whether the finding survives strict public availability.

In [ ]:
def metric(model,sample,column): return model_summary.query('model==@model and time_sample==@sample').iloc[0][column]
def future(model,sample,column): return future_results.query('model==@model and time_sample==@sample').iloc[0][column]
primary='rs_binary_10__economic_q_minus_1'; public='rs_binary_10__public_q_minus_2'; chars='characteristics_6__economic_q_minus_1'
decision=pd.DataFrame([
 {'test':'primary final-test target R2 positive','value':metric(primary,'final_test','pooled_variance_r2'),'passed':metric(primary,'final_test','pooled_variance_r2')>0},
 {'test':'primary exceeds characteristics in final test','value':metric(primary,'final_test','pooled_variance_r2')-metric(chars,'final_test','pooled_variance_r2'),'passed':metric(primary,'final_test','pooled_variance_r2')>metric(chars,'final_test','pooled_variance_r2')},
 {'test':'primary future-event interaction positive','value':future(primary,'final_test','interaction_slope'),'passed':future(primary,'final_test','interaction_slope')>0},
 {'test':'public-timing final-test target R2 positive','value':metric(public,'final_test','pooled_variance_r2'),'passed':metric(public,'final_test','pooled_variance_r2')>0},
 {'test':'public-timing future-event interaction positive','value':future(public,'final_test','interaction_slope'),'passed':future(public,'final_test','interaction_slope')>0}
])
decision.to_parquet(RESULTS/'ex_ante_embedding_decision_table.parquet',index=False); display(decision)
print('STOP: inspect results before behavioral-embedding and post-shock adjustment tests.')
